In [ ]:
import json
from pathlib import Path
from types import SimpleNamespace
from datetime import datetime, timezone
import sys
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import cumulative_trapezoid

REPO_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'ess').is_dir() and (p / 'data/cstr').is_dir()), None)  # Repository root, independent of the notebook working directory.
if REPO_ROOT is None:
    raise FileNotFoundError('Open this notebook from inside the repository.')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from ess.models import create_model, ControllerInput
from ess.data import load_trajectory, FeatureHistory
from ess.processes import CSTRPlant
from ess.reproducibility import seed_everything
import torch


In [ ]:
MODEL_DIR = REPO_ROOT / 'pretrained/cstr_leaky_mlp4'  # Bundled paper checkpoint, original normalization and metadata.
MODEL_PATH = MODEL_DIR / 'weights.h5'  # HDF5 neural-network weights; the MATLAB object is not required.
SCALER_DIR = MODEL_DIR / 'scalers'  # Original CSTR normalization files supplied alongside the model.
DEVICE = 'cpu'  # Device for autonomous inference.
PROGRESS_EVERY = 2000  # Samples between progress messages; 0 disables them.
SAVE_RESULTS = True  # Save metrics, trajectories and figures under outputs/evaluation.
OUTPUT_DIR = REPO_ROOT / 'outputs/evaluation/paper_model' / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')  # Separate outputs for each evaluation.


In [ ]:
def autonomous_rollout(model, trajectory, config, input_mean, input_scale, output_mean, output_scale, device, label):
    """Run the ANN alone with frozen weights and fresh plant/history state.

    At sample zero the applied action is zero. Each subsequent interval uses
    the preceding ANN prediction. There is no PID, sampling, blending or noise.
    Timing, input windows and process dynamics match the ESS evaluation path.
    """
    plant = CSTRPlant(config.sample_time_min)
    history = FeatureHistory(config)
    controller = ControllerInput(config.architecture, config.lstm_window, device)
    rows = np.empty((len(trajectory.reference), 8), dtype=float)
    action = 0.0
    model.eval()
    with torch.no_grad():
        for i, reference in enumerate(trajectory.reference):
            measurement = plant.step(action, trajectory.dq[i], trajectory.dcai[i])
            error = float(reference) - measurement
            features = history.update(error, action)
            prediction = controller.predict(model, (features - input_mean) / input_scale)
            next_action = prediction.item() * output_scale + output_mean
            rows[i] = [i * config.sample_time_min, reference, measurement, error,
                       action, next_action, trajectory.dcai[i], trajectory.dq[i]]
            if not np.isfinite(rows[i]).all():
                raise RuntimeError(f'{label}: non-finite trajectory at sample {i}; no finite IAE can be reported.')
            action = next_action
            if PROGRESS_EVERY and (i + 1) % PROGRESS_EVERY == 0:
                print(f'{label}: {i + 1}/{len(rows)} samples', flush=True)
    absolute_error = np.abs(rows[:, 3])
    cumulative_iae = cumulative_trapezoid(absolute_error, rows[:, 0], initial=0)
    metrics = dict(dataset=label, samples=len(rows), duration_min=float(rows[-1, 0]),
                   iae_trapezoid=float(cumulative_iae[-1]),
                   iae_rectangular_all_samples=float(config.sample_time_min * absolute_error.sum()),
                   iae_units='process-signal units * min')
    return rows, cumulative_iae, metrics


def plot_evaluation(rows, cumulative_iae, metrics):
    """Display process tracking, applied action and accumulated physical tracking IAE."""
    fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
    time = rows[:, 0]
    axes[0].plot(time, rows[:, 1], label='Reference', color='black', linestyle='--')
    axes[0].plot(time, rows[:, 2], label='ANN-controlled process')
    axes[0].set_ylabel('Process signal')
    axes[1].plot(time, rows[:, 4], label='Applied ANN action', color='tab:green')
    axes[1].set_ylabel('Action (physical units)')
    axes[2].plot(time, cumulative_iae, label='Cumulative IAE (trapezoidal)', color='tab:red')
    axes[2].set(ylabel='IAE (signal units·min)', xlabel='Time (min)')
    for axis in axes:
        axis.legend()
        axis.grid(True, alpha=.3)
    fig.suptitle(f"{metrics['dataset'].title()} trajectory — ANN alone — IAE = {metrics['iae_trapezoid']:.6f}")
    fig.tight_layout(rect=(0, 0, 1, .96))
    if SAVE_RESULTS:
        fig.savefig(OUTPUT_DIR / f"{metrics['dataset']}_tracking.png", dpi=150)
    plt.show()
    plt.close(fig)


In [ ]:
import h5py
import hashlib
from ess import ESSConfig

metadata = json.loads((MODEL_DIR / 'metadata.json').read_text())
config = ESSConfig(device=DEVICE, **metadata['evaluation_config'])
input_mean = np.loadtxt(SCALER_DIR / 'input_mean.csv').reshape(4)
input_scale = np.loadtxt(SCALER_DIR / 'input_scale.csv').reshape(4)
output_mean = float(np.loadtxt(SCALER_DIR / 'output_mean.csv'))
output_scale = float(np.loadtxt(SCALER_DIR / 'output_scale.csv'))
seed_everything(config.seed, config.deterministic)
device = torch.device(DEVICE)
model = create_model(config.architecture, device)
with h5py.File(MODEL_PATH, 'r') as handle:
    weights = {key: torch.as_tensor(handle[key][...], dtype=torch.float32, device=device) for key in handle.keys()}
model.load_state_dict(weights, strict=True)
model.requires_grad_(False)
model.eval()
model_sha256 = hashlib.sha256(MODEL_PATH.read_bytes()).hexdigest()
if model_sha256 != metadata['weights_sha256']:
    raise ValueError('Paper-model checksum does not match metadata.json.')
if SAVE_RESULTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
print(f'Paper model: {MODEL_PATH}\nSHA256: {model_sha256}')
print('Architecture: LeakyMLP4 (4–256–128–64–1), LeakyReLU slope 0.01.')
print('Original CSRT1_4i scalers; 300-sample error window; cumulative-error retention 1.0.')
print('ANN alone; no PID control, blending, noise, scaler fitting or weight updates.')


In [ ]:
results = {}
metrics_rows = []
data_dir = REPO_ROOT / config.data_dir
for label, data_file, excitation_file in (
    ('training', config.training_file, config.training_excitation),
    ('validation', config.validation_file, config.validation_excitation),
):
    trajectory = load_trajectory(data_dir / data_file, data_dir / excitation_file, config)
    print(f'Running {label}: {len(trajectory.reference)} samples...', flush=True)
    rows, cumulative_iae, metrics = autonomous_rollout(
        model, trajectory, config, input_mean, input_scale, output_mean, output_scale, device, label)
    results[label] = (rows, cumulative_iae)
    metrics_rows.append(metrics)
    print(f"{label}: IAE (trapezoid) = {metrics['iae_trapezoid']:.6f}; "
          f"IAE (dt × sum) = {metrics['iae_rectangular_all_samples']:.6f}", flush=True)
    if SAVE_RESULTS:
        np.savetxt(OUTPUT_DIR / f'{label}_rollout.csv', np.column_stack((rows, cumulative_iae)),
                   delimiter=',', comments='',
                   header='time_min,reference,process_output,error,applied_action,ann_action_next,dCAi,dQ,cumulative_iae')
    plot_evaluation(rows, cumulative_iae, metrics)


In [ ]:
paper_iae = {'training': 345.0, 'validation': 272.0}
print('\nPaper-checkpoint ANN-only tracking IAE — signal units·minutes')
print(f"{'Dataset':<14} {'IAE trapezoid':>16} {'IAE dt*sum':>16} ")
for metric in metrics_rows:
    metric['paper_iae'] = paper_iae[metric['dataset']]
    metric['difference_percent'] = 100 * (metric['iae_trapezoid'] / metric['paper_iae'] - 1)
    print(f"{metric['dataset']:<14} {metric['iae_trapezoid']:>16.6f} "
          f"{metric['iae_rectangular_all_samples']:>16.6f} "
          )
print('\nReference: Journal_ISA26, Table II, ESS LeakyMLP4 (308e).')
print('Comparison uses the repository linearized CSTR and the original CSTR scaler files.')
print('The supplied HDF5 does not encode an epoch number or preprocessing metadata; settings are taken from the original CSTR configuration.')
if SAVE_RESULTS:
    report = dict(model=str(MODEL_PATH.relative_to(REPO_ROOT)), model_sha256=model_sha256,
                  device=str(device), architecture=config.architecture, sample_time_min=config.sample_time_min,
                  error_window=config.error_window, error_retention=config.error_retention,
                  dataset_fraction=config.dataset_fraction,
                  scaler_source='pretrained/cstr_leaky_mlp4/scalers; provenance in metadata.json',
                  input_mean=input_mean.tolist(), input_scale=input_scale.tolist(),
                  output_mean=output_mean, output_scale=output_scale, metrics=metrics_rows)
    (OUTPUT_DIR / 'metrics.json').write_text(json.dumps(report, indent=2) + '\n')
    import csv
    with (OUTPUT_DIR / 'metrics.csv').open('w', newline='') as stream:
        writer = csv.DictWriter(stream, fieldnames=list(metrics_rows[0]))
        writer.writeheader()
        writer.writerows(metrics_rows)
    print(f'\nPaper-model evaluation files: {OUTPUT_DIR}')
